# Varian 2 — Validasi K dengan Silhouette Score

## Proyek Akhir Pembelajaran Mesin (INF62325)

### Analisis Properti Rumah dengan Implementasi Algoritma K-Means Berdasarkan Karakteristik dan Harga di Wilayah Jabodetabek

| Identitas | Keterangan |
|---|---|
| Nama | Najwa Ramadhan |
| NIM | 2488010001 |
| Kelas | Informatika 5A |
| Anggota | 2 |
| Varian | Parameter — Validasi K |
| Dataset | Jabodetabek House Price |
| Algoritma | K-Means Clustering |

## Tujuan Eksperimen

Eksperimen ini bertujuan untuk menentukan jumlah cluster
yang sesuai pada algoritma K-Means melalui pengujian
beberapa nilai K menggunakan Silhouette Score.

Hasil eksperimen dibandingkan dengan metode Elbow
pada Varian 1 untuk mengetahui perbedaan kualitas
pengelompokan properti rumah di wilayah Jabodetabek.

Metrik pendukung yang digunakan adalah Inertia
dan Davies-Bouldin Index.

## 0. Persiapan — **jalankan setiap kali notebook dibuka**

Colab menghapus seluruh berkas setiap kali runtime dimulai ulang. Tiga sel di bawah memasang pustaka, menulis `alat.py`, dan menyiapkan dua fungsi pembantu. Ketiganya harus dijalankan lebih dulu, apa pun bagian yang Anda kerjakan di notebook ini.

`alat.py` berisi fungsi yang dipakai ketiga varian. **Jangan mengubahnya sendiri-sendiri:** kalau satu anggota mengubah isi berkas ini, ketiga varian berhenti sebanding dan tabel perbandingan menjadi tidak sah.

**0.1 — Import library dan konfigurasi**

In [1]:
import ast, json, os, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, sklearn

warnings.filterwarnings('ignore')

pd.set_option('display.width', 170)
pd.set_option('display.max_columns', 40)

for d in ('data', 'data/hasil_varian', 'model', 'gambar'):
    os.makedirs(d, exist_ok=True)

SEED = 42
ID_COL = 'ads_id'
IQR_PENGALI = 1.5

print('Siap. scikit-learn', sklearn.__version__)
print('Catat versi ini untuk requirements.txt.')

Siap. scikit-learn 1.6.1
Catat versi ini untuk requirements.txt.


In [2]:
%%writefile alat.py
"""alat.py -- fungsi bersama ketiga varian. Jangan diubah sendiri-sendiri."""
from __future__ import annotations

import json
import os

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler

SEED = 42
RENTANG_K = range(2, 11)
BERKAS_BERSIH = 'data/data_bersih.csv'
FOLDER_HASIL = 'data/hasil_varian'

# Label dan satuan untuk antarmuka aplikasi (ketentuan Tahap 3: setiap
# masukan harus punya label dan satuan yang dipahami orang luar).
LABEL = {
    'BALANCE': ('Saldo rata-rata', 'USD'),
    'BALANCE_FREQUENCY': ('Keseringan saldo diperbarui', 'skala 0-1'),
    'PURCHASES': ('Total pembelian', 'USD'),
    'ONEOFF_PURCHASES': ('Pembelian sekali bayar', 'USD'),
    'INSTALLMENTS_PURCHASES': ('Pembelian cicilan', 'USD'),
    'CASH_ADVANCE': ('Penarikan tunai', 'USD'),
    'PURCHASES_FREQUENCY': ('Keseringan berbelanja', 'skala 0-1'),
    'ONEOFF_PURCHASES_FREQUENCY': ('Keseringan belanja sekali bayar', 'skala 0-1'),
    'PURCHASES_INSTALLMENTS_FREQUENCY': ('Keseringan belanja cicilan', 'skala 0-1'),
    'CASH_ADVANCE_FREQUENCY': ('Keseringan menarik tunai', 'skala 0-1'),
    'CASH_ADVANCE_TRX': ('Jumlah transaksi tarik tunai', 'kali'),
    'PURCHASES_TRX': ('Jumlah transaksi belanja', 'kali'),
    'CREDIT_LIMIT': ('Limit kartu', 'USD'),
    'PAYMENTS': ('Total pembayaran', 'USD'),
    'MINIMUM_PAYMENTS': ('Pembayaran minimum', 'USD'),
    'PRC_FULL_PAYMENT': ('Proporsi pelunasan penuh', 'skala 0-1'),
    'TENURE': ('Lama berlangganan', 'bulan'),
}


def label_satuan(kolom: str) -> tuple[str, str]:
    return LABEL.get(kolom, (kolom.replace('_', ' ').title(), ''))


def muat_bersih(fitur: list[str] | None = None) -> pd.DataFrame:
    """Membaca data bersih kelompok. Kolom identitas selalu dibuang."""
    df = pd.read_csv(BERKAS_BERSIH)
    # Hanya kolom numerik yang menjadi fitur pembentuk klaster.
    # Kolom identitas seperti CUST_ID tidak membawa informasi kemiripan.
    df = df.select_dtypes(include='number')
    return df[fitur] if fitur else df


def skalakan(df: pd.DataFrame):
    sc = StandardScaler()
    return sc.fit_transform(df), sc


def jalankan_k(X_scaled, rentang=RENTANG_K) -> pd.DataFrame:
    """Menjalankan K-Means untuk tiap k, mengembalikan tabel metrik."""
    baris = []
    for k in rentang:
        km = KMeans(n_clusters=k, init='k-means++', n_init=20,
                    random_state=SEED)
        lab = km.fit_predict(X_scaled)
        baris.append({
            'k': k,
            'inertia': float(km.inertia_),
            'silhouette': float(silhouette_score(X_scaled, lab)),
            'davies_bouldin': float(davies_bouldin_score(X_scaled, lab)),
        })
    return pd.DataFrame(baris)


def titik_siku(ks, inertia) -> int:
    """Titik siku = jarak tegak lurus terjauh dari garis ujung ke ujung."""
    x = np.asarray(ks, dtype=float)
    y = np.asarray(inertia, dtype=float)
    x_n = (x - x.min()) / (x.max() - x.min())
    y_n = (y - y.min()) / (y.max() - y.min())
    x1, y1, x2, y2 = x_n[0], y_n[0], x_n[-1], y_n[-1]
    jarak = np.abs((y2 - y1) * x_n - (x2 - x1) * y_n + x2 * y1 - y2 * x1)
    jarak /= np.hypot(y2 - y1, x2 - x1)
    return int(x[int(np.argmax(jarak))])


def profil_relatif(df: pd.DataFrame, label) -> pd.DataFrame:
    """Simpangan rata-rata tiap klaster terhadap rata-rata seluruh data,
    dalam satuan simpangan baku. Baris = fitur, kolom = klaster."""
    d = df.copy()
    d['cluster'] = label
    rata = d.drop(columns='cluster').mean()
    sd = d.drop(columns='cluster').std().replace(0, np.nan)
    return ((d.groupby('cluster').mean() - rata) / sd).round(2).T


def nama_otomatis(profil: pd.DataFrame) -> dict[str, str]:
    """Dua fitur dengan simpangan terbesar secara mutlak, beserta arahnya.
    WAJIB diganti mahasiswa dengan nama yang bermakna."""
    nama = {}
    for c in profil.columns:
        kol = profil[c]
        kuat = kol.reindex(kol.abs().sort_values(ascending=False).index)[:2]
        nama[str(int(c))] = ', '.join(
            f'{n} {"tinggi" if v > 0 else "rendah"}' for n, v in kuat.items())
    return nama


def simpan_hasil(nomor: int, isi: dict) -> str:
    os.makedirs(FOLDER_HASIL, exist_ok=True)
    jalur = f'{FOLDER_HASIL}/varian{nomor}.json'
    with open(jalur, 'w', encoding='utf-8') as f:
        json.dump(isi, f, indent=2, ensure_ascii=False)
    return jalur


def baca_hasil(nomor: int) -> dict | None:
    jalur = f'{FOLDER_HASIL}/varian{nomor}.json'
    if not os.path.exists(jalur):
        return None
    with open(jalur, encoding='utf-8') as f:
        return json.load(f)

Writing alat.py


**0.3 — Fungsi upload berkas**

In [4]:
def ambil_berkas():
    """Unggah berkas dari komputer, lalu letakkan di folder yang benar."""
    from google.colab import files
    for nama in files.upload():
        if nama.startswith('varian') and nama.endswith('.json'):
            tujuan = 'data/hasil_varian/' + nama
        elif nama.endswith('.pkl') or nama == 'segmen.json':
            tujuan = 'model/' + nama
        elif nama.endswith('.png'):
            tujuan = 'gambar/' + nama
        else:
            tujuan = 'data/' + nama
        shutil.move(nama, tujuan)
        print('->', tujuan)


def perlu(*jalur):
    """Minta unggahan HANYA untuk berkas yang belum ada di sesi ini."""
    kurang = [j for j in jalur if not os.path.exists(j)]
    if not kurang:
        print('Lengkap:', ', '.join(jalur))
        return
    print('Belum ada:', ', '.join(kurang))
    print('Unggah berkas tersebut sekarang.')
    ambil_berkas()
    masih = [j for j in jalur if not os.path.exists(j)]
    if masih:
        raise FileNotFoundError('Masih kurang: ' + ', '.join(masih))
    print('Lengkap.')


print('ambil_berkas() dan perlu() siap dipakai.')

ambil_berkas() dan perlu() siap dipakai.


# Bagian 2 — Varian 1 — Penentuan k dengan Metode Elbow

**Proyek Akhir Pembelajaran Mesin (INF62325) · Cabang C — Klastering K-Means**
**S1 Informatika · UIN Siber Syekh Nurjati Cirebon**

| | |
|---|---|
| Pertemuan | dikerjakan setelah Pertemuan 12, dipresentasikan Pertemuan 14 |
| Tahap resmi | Tahap 2 — Varian Eksperimen 1 |
| Dikerjakan oleh | **Anggota 1** saja |
| Berkas yang dibutuhkan | `data_bersih.csv` dari Bagian 1 |
| Luaran wajib | `data/hasil_varian/varian1.json`, `gambar/varian1_elbow.png` |
| Dinilai pada | Progres 2, Pertemuan 14, **35% perorangan** |
| Tenggat | selesai sebelum Pertemuan 14 |

| | |
|---|---|
| Kelompok | _(isi)_ |
| Paket dataset | _(isi sesuai penetapan dosen)_ |
| Nama dan NIM | _(isi)_ |

---

## Panduan Bagian 2

> **Ketentuan Panduan Proyek Akhir, Cabang C, Varian 1:** *Metode Elbow untuk k = 2 sampai 10, lalu tentukan titik sikunya.*

Bagian ini dikerjakan **Anggota 1 saja**, dan dinilai **perorangan**. Jangan menjalankan Bagian milik anggota lain: riwayat commit dipakai sebagai bukti siapa mengerjakan apa.

**Urutan kerja**

1. Jalankan 2.0 (persiapan).
2. Jalankan 2.1. Bila `data_bersih.csv` belum ada di sesi Anda, Colab akan meminta unggahannya.
3. Jalankan 2.2. Perhatikan nilai inertia tiap k dan posisi titik sikunya.
4. **Tulis sendiri** jawaban di 2.3. Interpretasi inilah yang paling banyak membedakan nilai antaranggota.
5. Jalankan 2.4 untuk mengunduh hasil, lalu commit dengan akun GitHub **Anda sendiri**.
6. Kirim `varian1.json` ke Anggota 2. Varian 2 membutuhkannya untuk membandingkan kesimpulan.

**Yang perlu dipahami lebih dulu**

Inertia adalah jumlah kuadrat jarak setiap titik ke pusat klasternya. Nilainya **selalu** turun ketika k dinaikkan, jadi "inertia terkecil" bukan jawaban — kalau k disamakan dengan jumlah baris, inertia menjadi nol dan hasilnya tidak berguna. Yang dicari adalah titik di mana penurunan mulai melandai, karena di situ penambahan klaster berhenti memberi manfaat berarti.

Menunjuk siku dengan mata sering berbeda antarorang. Sel 2.2 memakai cara yang dapat diperiksa ulang: **jarak tegak lurus terjauh** dari garis yang menghubungkan titik k terkecil ke titik k terbesar. Yang dinilai bukan besarnya angka, melainkan apakah titik siku ditentukan dengan cara yang dapat dijelaskan.

Silhouette ikut dihitung di sini, tetapi hanya sebagai metrik pendukung. Dasar Bagian ini adalah inertia.

---

## 2.1 Mengambil berkas yang diperlukan

Sel ini memeriksa berkas yang Anda butuhkan. Yang sudah ada dilewati; yang belum ada akan diminta untuk diunggah. Aman dijalankan berulang kali.

In [ ]:
perlu('data/data_bersih.csv')

Belum ada: data/data_bersih.csv
Unggah berkas tersebut sekarang.


Saving data_bersih.csv to data_bersih.csv
-> data/data_bersih.csv
Lengkap.


## 2.2 Menjalankan varian 1

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

from alat import (jalankan_k, muat_bersih, simpan_hasil, skalakan, titik_siku)

print('=' * 70)
print('VARIAN 1 - PENENTUAN k DENGAN METODE ELBOW')
print('=' * 70)

df = muat_bersih()
X, _ = skalakan(df)
print(f'Data bersih kelompok: {df.shape[0]} baris, {df.shape[1]} fitur\n')

tabel = jalankan_k(X)
tabel['penurunan'] = tabel['inertia'].diff().abs()
tabel['penurunan_persen'] = (tabel['penurunan'] / tabel['inertia'].shift(1) * 100)
print(tabel[['k', 'inertia', 'penurunan', 'penurunan_persen', 'silhouette']]
      .round(3).to_string(index=False))

k_siku = titik_siku(tabel['k'], tabel['inertia'])
sil_siku = float(tabel.loc[tabel['k'] == k_siku, 'silhouette'].iloc[0])
inertia_siku = float(tabel.loc[tabel['k'] == k_siku, 'inertia'].iloc[0])

print(f'\nTitik siku  : k = {k_siku}')
print(f'Inertia     : {inertia_siku:,.2f}')
print(f'Silhouette  : {sil_siku:.4f}  (metrik pendukung, bukan dasar varian ini)')
print('\nCara menentukannya: titik siku diambil pada jarak tegak lurus terjauh')
print('dari garis yang menghubungkan k terkecil dan k terbesar. Cara ini')
print('menggantikan penilaian dengan mata, yang sering berbeda antarorang.')

plt.figure(figsize=(8, 5))
plt.plot(tabel['k'], tabel['inertia'], marker='o')
plt.axvline(k_siku, ls='--', lw=1, color='grey')
plt.annotate(f'titik siku k={k_siku}', xy=(k_siku, inertia_siku),
             xytext=(k_siku + 0.6, inertia_siku * 1.08),
             arrowprops=dict(arrowstyle='->', lw=1))
plt.xlabel('Jumlah klaster (k)'); plt.ylabel('Inertia (WCSS)')
plt.title('Varian 1 - Metode Elbow')
plt.tight_layout(); plt.savefig('gambar/varian1_elbow.png', dpi=110); plt.close()

jalur = simpan_hasil(1, {
    'varian': 1,
    'nama': 'Penentuan k (Metode Elbow)',
    'yang_diubah': 'Jumlah klaster ditentukan dari titik siku kurva inertia',
    'fitur_dipakai': df.columns.tolist(),
    'k_terpilih': k_siku,
    'metrik_utama': {'silhouette': round(sil_siku, 4)},
    'metrik_pendukung': {'inertia': round(inertia_siku, 2)},
    'tabel_k': tabel[['k', 'inertia', 'silhouette', 'davies_bouldin']]
                .round(4).to_dict(orient='records'),
})
print(f'\nTersimpan: {jalur}  dan  gambar/varian1_elbow.png')
print('Commit berkas ini dengan akun GitHub Anda sendiri.')

VARIAN 1 - PENENTUAN k DENGAN METODE ELBOW
Data bersih kelompok: 3000 baris, 17 fitur

 k   inertia  penurunan  penurunan_persen  silhouette
 2 32444.565        NaN               NaN       0.346
 3 22437.154  10007.411            30.845       0.390
 4 18097.207   4339.947            19.343       0.410
 5 16040.973   2056.234            11.362       0.355
 6 15208.428    832.545             5.190       0.321
 7 14573.742    634.686             4.173       0.302
 8 13964.663    609.079             4.179       0.248
 9 13313.634    651.029             4.662       0.254
10 12756.070    557.564             4.188       0.227

Titik siku  : k = 4
Inertia     : 18,097.21
Silhouette  : 0.4096  (metrik pendukung, bukan dasar varian ini)

Cara menentukannya: titik siku diambil pada jarak tegak lurus terjauh
dari garis yang menghubungkan k terkecil dan k terbesar. Cara ini
menggantikan penilaian dengan mata, yang sering berbeda antarorang.

Tersimpan: data/hasil_varian/varian1.json  dan  gambar/va

## 2.3 Yang harus Anda tulis sendiri

Rubrik varian perorangan menilai tiga hal: **ketepatan implementasi**, **kebenaran evaluasi**, dan **kedalaman interpretasi**. Butir ketiga tidak dapat dihasilkan oleh kode, dan di situlah nilai paling banyak hilang.

1. Apa arti titik siku ini dalam konteks kasus kelompok Anda? Jawab dengan menyebut jumlah segmen, bukan hanya angka k.
2. Seberapa tegas sikunya? Bila tumpul, nilai k mana saja yang masuk akal, dan apa pertimbangan memilih salah satunya?
3. Apa yang berubah pada aplikasi web bila k Anda yang dipakai?

_(jawab di sini — klik dua kali pada sel untuk menyuntingnya)_

## 2.4 Unduh, commit, dan teruskan

In [ ]:
# Unduh hasil Bagian 2, lalu commit ke repositori dengan akun GitHub
# ANDA SENDIRI. Riwayat commit adalah bukti penilaian perorangan.
from google.colab import files
files.download('data/hasil_varian/varian1.json')
files.download('gambar/varian1_elbow.png')
print()
print('Langkah berikutnya:')
print('  1. Commit kedua berkas di atas dengan akun GitHub Anda sendiri.')
print('  2. File -> Download -> Download .ipynb, beri nama 03_varian_1.ipynb,')
print('     lalu commit ke folder notebook/.')
print('  3. Teruskan `varian1.json` ke Anggota 2.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Langkah berikutnya:
  1. Commit kedua berkas di atas dengan akun GitHub Anda sendiri.
  2. File -> Download -> Download .ipynb, beri nama 03_varian_1.ipynb,
     lalu commit ke folder notebook/.
  3. Teruskan `varian1.json` ke Anggota 2.


## Checklist Bagian 2

- [ ] `varian1.json` ter-commit di `data/hasil_varian/`
- [ ] `varian1_elbow.png` ter-commit di `gambar/`
- [ ] Notebook ini diunduh, diberi nama `03_varian_1.ipynb`, dan ter-commit di `notebook/`
- [ ] Commit dilakukan dengan akun GitHub Anggota 1 sendiri
- [ ] Ketiga jawaban di 2.3 sudah ditulis
- [ ] `varian1.json` sudah dikirim ke Anggota 2

# Bagian 3 — Varian 2 — Validasi k dengan Silhouette Score

**Proyek Akhir Pembelajaran Mesin (INF62325) · Cabang C — Klastering K-Means**
**S1 Informatika · UIN Siber Syekh Nurjati Cirebon**

| | |
|---|---|
| Pertemuan | dikerjakan setelah Pertemuan 12, dipresentasikan Pertemuan 14 |
| Tahap resmi | Tahap 2 — Varian Eksperimen 2 |
| Dikerjakan oleh | **Anggota 2** saja |
| Berkas yang dibutuhkan | `data_bersih.csv` dari Bagian 1 dan `varian1.json` dari Anggota 1 |
| Luaran wajib | `data/hasil_varian/varian2.json`, `gambar/varian2_silhouette.png` |
| Dinilai pada | Progres 2, Pertemuan 14, **35% perorangan** |
| Tenggat | selesai sebelum Pertemuan 14 |

| | |
|---|---|
| Kelompok | _(isi)_ |
| Paket dataset | _(isi sesuai penetapan dosen)_ |
| Nama dan NIM | _(isi)_ |

---

## Panduan Bagian 3

> **Ketentuan Panduan Proyek Akhir, Cabang C, Varian 2:** *Silhouette Score untuk tiap k. Jika kesimpulannya berbeda dari Elbow, jelaskan mengapa.*

Bagian ini dikerjakan **Anggota 2 saja**, dan dinilai **perorangan**.

**Urutan kerja**

1. Jalankan 3.0 (persiapan).
2. Jalankan 3.1. Anda membutuhkan dua berkas: `data_bersih.csv` dan `varian1.json` dari Anggota 1. Minta berkasnya lebih dulu kalau belum ada.
3. Jalankan 3.2.
4. **Tulis sendiri** jawaban di 3.3.
5. Jalankan 3.4, commit dengan akun Anda sendiri, lalu kirim `varian2.json` ke Anggota 3.

**Yang perlu dipahami lebih dulu**

Silhouette Score mengukur, untuk setiap titik, seberapa dekat ia ke klasternya sendiri dibanding ke klaster terdekat berikutnya. Nilainya antara −1 dan 1. Berbeda dari inertia, nilainya **tidak** otomatis membaik ketika k dinaikkan, jadi nilai tertinggi memang dapat dipakai sebagai pertimbangan.

Patokan tafsiran yang dipakai mata kuliah ini:

| Nilai | Tafsiran |
|---|---|
| di atas 0,50 | pemisahan klaster jelas |
| 0,25 sampai 0,50 | struktur lemah, tetapi masih dapat dipertanggungjawabkan |
| di bawah 0,25 | struktur hampir tidak ada |

Pada data pelanggan nyata, angka 0,20 sampai 0,35 adalah hal biasa. Angka rendah bukan berarti pekerjaan Anda salah; yang salah adalah melaporkannya seolah-olah pemisahannya jelas.

**Satu jebakan yang sering terjadi.** Silhouette hampir selalu tertinggi di k=2, karena membagi data menjadi dua bagian besar paling mudah dipisahkan. Nilai itu biasanya tidak dipakai, sebab dua segmen tidak memberi informasi yang berguna untuk tindakan. Kalau k Anda bukan nilai dengan Silhouette tertinggi, **itu justru perlu dijelaskan**, bukan disembunyikan.

Bila kesimpulan Anda berbeda dari Bagian 2, sel di bawah menyusun kerangka penjelasannya secara otomatis — tetapi Anda tetap harus menuliskannya dengan kalimat Anda sendiri.

---

## 3.1 Mengambil berkas yang diperlukan

Sel ini memeriksa berkas yang Anda butuhkan. Yang sudah ada dilewati; yang belum ada akan diminta untuk diunggah. Aman dijalankan berulang kali.

## 3.1 Mengambil Berkas yang Diperlukan

Eksperimen Varian 2 menggunakan dataset bersih
hasil praproses kelompok dan hasil eksperimen
metode Elbow dari Varian 1.

Penggunaan dataset yang sama bertujuan untuk
menjaga konsistensi dan memastikan hasil kedua
varian dapat dibandingkan secara objektif.

In [6]:
perlu(
    'data/data_bersih.csv',
    'data/hasil_varian/varian1.json'
)

Belum ada: data/data_bersih.csv
Unggah berkas tersebut sekarang.


Saving varian1.json to varian1.json
Saving data_bersih.csv to data_bersih.csv
-> data/hasil_varian/varian1.json
-> data/data_bersih.csv
Lengkap.


In [7]:
from alat import muat_bersih, baca_hasil

df = muat_bersih()
v1 = baca_hasil(1)

print("Jumlah data:", df.shape[0])
print("Jumlah fitur:", df.shape[1])

print("\nFitur yang digunakan:")
for kolom in df.columns:
    print("-", kolom)

print("\nMissing value:", df.isna().sum().sum())

print("\nK terpilih Varian 1:", v1["k_terpilih"])

assert df.columns.tolist() == v1["fitur_dipakai"], \
    "Fitur Varian 1 dan Varian 2 berbeda!"

assert np.isfinite(df.to_numpy()).all(), \
    "Dataset mengandung nilai tidak valid!"

print("\nValidasi berhasil. Data siap digunakan.")

Jumlah data: 3470
Jumlah fitur: 6

Fitur yang digunakan:
- price_in_rp
- land_size_m2
- building_size_m2
- bedrooms
- bathrooms
- floors

Missing value: 0

K terpilih Varian 1: 4

Validasi berhasil. Data siap digunakan.


## 3.2 Menjalankan varian 2

In [8]:
import matplotlib
import matplotlib.pyplot as plt

from alat import baca_hasil, jalankan_k, muat_bersih, simpan_hasil, skalakan

print('=' * 70)
print('VARIAN 2 - VALIDASI k DENGAN SILHOUETTE SCORE')
print('=' * 70)

df = muat_bersih()
X, _ = skalakan(df)
print(f'Data bersih kelompok: {df.shape[0]} baris, {df.shape[1]} fitur\n')

tabel = jalankan_k(X)
print(tabel[['k', 'silhouette', 'davies_bouldin', 'inertia']]
      .round(4).to_string(index=False))

k_terbaik = int(tabel.loc[tabel['silhouette'].idxmax(), 'k'])
sil_terbaik = float(tabel['silhouette'].max())

print(f'\nSilhouette tertinggi : k = {k_terbaik}  ({sil_terbaik:.4f})')
if sil_terbaik >= 0.5:
    tafsir = 'pemisahan klaster jelas'
elif sil_terbaik >= 0.25:
    tafsir = 'struktur lemah tetapi masih dapat dipertanggungjawabkan'
else:
    tafsir = 'struktur klaster hampir tidak ada, pilihan fitur perlu ditinjau'
print(f'Tafsiran nilainya    : {tafsir}')

# --- Bandingkan dengan kesimpulan Varian 1 ---
v1 = baca_hasil(1)
print('\n' + '-' * 70)
if v1 is None:
    print('Hasil Varian 1 belum ada. Jalankan varian1_elbow.py lebih dulu')
    print('agar perbandingan antarvarian dapat disusun.')
    k_elbow, beda, penjelasan = None, None, None
else:
    k_elbow = v1['k_terpilih']
    beda = k_elbow != k_terbaik
    print(f'Kesimpulan Varian 1 (Elbow)      : k = {k_elbow}')
    print(f'Kesimpulan Varian 2 (Silhouette) : k = {k_terbaik}')
    if not beda:
        penjelasan = ('Kedua varian menunjuk nilai k yang sama, sehingga '
                      'pilihan k berdiri di atas dua dasar yang saling '
                      'menguatkan.')
    else:
        sil_elbow = float(tabel.loc[tabel['k'] == k_elbow, 'silhouette'].iloc[0])
        selisih = sil_terbaik - sil_elbow
        penjelasan = (
            f'Kedua varian berbeda. Elbow mengukur seberapa rapat anggota '
            f'terhadap pusat klasternya sendiri, sedangkan Silhouette juga '
            f'memperhitungkan jarak ke klaster tetangga. Inertia selalu turun '
            f'saat k bertambah, sehingga titik sikunya bergantung pada bentuk '
            f'kurva, bukan pada mutu pemisahan. Pada k={k_elbow} Silhouette '
            f'bernilai {sil_elbow:.4f}, selisih {selisih:.4f} dari nilai '
            f'tertinggi di k={k_terbaik}.')
        if k_terbaik == 2:
            penjelasan += (' Perlu dicatat, Silhouette sering memuncak di k=2 '
                           'karena dua klaster paling mudah dipisahkan, '
                           'padahal dua segmen jarang berguna untuk '
                           'pengambilan keputusan.')
    print('\n' + penjelasan)
print('-' * 70)

plt.figure(figsize=(12, 4.5))
plt.subplot(1, 2, 1)
plt.plot(tabel['k'], tabel['silhouette'], marker='o', color='green')
plt.axvline(k_terbaik, ls='--', lw=1, color='grey')
if k_elbow:
    plt.axvline(k_elbow, ls=':', lw=1.5, color='tab:blue')
plt.xlabel('k'); plt.ylabel('Silhouette Score'); plt.title('Varian 2 - Silhouette')
plt.subplot(1, 2, 2)
plt.plot(tabel['k'], tabel['davies_bouldin'], marker='o', color='red')
plt.xlabel('k'); plt.ylabel('Davies-Bouldin'); plt.title('Metrik pendukung')
plt.tight_layout(); plt.savefig('gambar/varian2_silhouette.png', dpi=110); plt.close()

jalur = simpan_hasil(2, {
    'varian': 2,
    'nama': 'Validasi k (Silhouette Score)',
    'yang_diubah': 'Jumlah klaster ditentukan dari Silhouette tertinggi',
    'fitur_dipakai': df.columns.tolist(),
    'k_terpilih': k_terbaik,
    'k_menurut_elbow': k_elbow,
    'berbeda_dari_elbow': beda,
    'penjelasan_perbedaan': penjelasan,
    'tafsiran': tafsir,
    'metrik_utama': {'silhouette': round(sil_terbaik, 4)},
    'metrik_pendukung': {
        'inertia': round(float(tabel.loc[tabel['k'] == k_terbaik, 'inertia'].iloc[0]), 2),
        'davies_bouldin': round(float(tabel.loc[tabel['k'] == k_terbaik, 'davies_bouldin'].iloc[0]), 4),
    },
    'tabel_k': tabel[['k', 'inertia', 'silhouette', 'davies_bouldin']]
                .round(4).to_dict(orient='records'),
})
print(f'\nTersimpan: {jalur}  dan  gambar/varian2_silhouette.png')
print('Commit berkas ini dengan akun GitHub Anda sendiri.')

VARIAN 2 - VALIDASI k DENGAN SILHOUETTE SCORE
Data bersih kelompok: 3470 baris, 6 fitur

 k  silhouette  davies_bouldin   inertia
 2      0.4834          0.8806 9915.2324
 3      0.3994          0.8887 6950.7500
 4      0.3714          1.0896 5708.8196
 5      0.3645          1.2212 5106.7052
 6      0.3720          1.1604 4630.2684
 7      0.3623          1.2198 4256.7722
 8      0.3507          1.2606 3909.3705
 9      0.3513          1.2161 3654.4491
10      0.3556          1.2335 3456.0970

Silhouette tertinggi : k = 2  (0.4834)
Tafsiran nilainya    : struktur lemah tetapi masih dapat dipertanggungjawabkan

----------------------------------------------------------------------
Kesimpulan Varian 1 (Elbow)      : k = 4
Kesimpulan Varian 2 (Silhouette) : k = 2

Kedua varian berbeda. Elbow mengukur seberapa rapat anggota terhadap pusat klasternya sendiri, sedangkan Silhouette juga memperhitungkan jarak ke klaster tetangga. Inertia selalu turun saat k bertambah, sehingga titik sikunya b

## 3.3 Yang harus Anda tulis sendiri

Rubrik varian perorangan menilai tiga hal: **ketepatan implementasi**, **kebenaran evaluasi**, dan **kedalaman interpretasi**. Butir ketiga tidak dapat dihasilkan oleh kode, dan di situlah nilai paling banyak hilang.

1. Apakah kesimpulan Anda sama dengan Elbow di Bagian 2? Bila berbeda, jelaskan dengan kalimat Anda sendiri mengapa dua metrik dapat menunjuk k yang berbeda.
2. Bila Silhouette tertinggi ada di k=2, mengapa nilai itu tidak dipakai?
3. Apa arti angka Silhouette ini dalam konteks kasus kelompok Anda? Pakai patokan tafsiran di atas, jangan hanya menyebut angkanya.

## **3.3 Interpretasi Hasil Eksperimen**

### **1. Apakah kesimpulan Varian 2 sama dengan metode Elbow pada Varian 1? Jika berbeda, mengapa?**

Berdasarkan hasil eksperimen, terdapat perbedaan antara hasil Varian 1 dan Varian 2. Metode Silhouette Score pada Varian 2 menghasilkan jumlah cluster terbaik pada K=2 dengan nilai sebesar 0,4834, sedangkan metode Elbow pada Varian 1 menghasilkan K=4 dengan nilai Silhouette Score sebesar 0,3714.

Perbedaan ini terjadi karena kedua metode memiliki cara penilaian yang berbeda. Metode Elbow melihat perubahan nilai inertia dan menentukan titik ketika penurunannya mulai melandai. Sementara itu, Silhouette Score menilai seberapa mirip suatu data dengan anggota dalam clusternya sendiri dan seberapa berbeda data tersebut dengan cluster lainnya.

Dalam kasus pengelompokan properti rumah di Jabodetabek, K=2 menghasilkan pemisahan cluster yang lebih baik berdasarkan Silhouette Score, sedangkan K=4 berpotensi memberikan pengelompokan properti yang lebih detail. Jadi, perbedaan hasil tersebut bukan karena adanya kesalahan dalam eksperimen, melainkan karena kedua metode menggunakan cara penilaian yang berbeda.

### **2. Jika Silhouette Score tertinggi berada pada K=2, mengapa nilai tersebut tidak langsung digunakan sebagai model akhir?**

Meskipun K=2 memiliki nilai Silhouette Score tertinggi, bukan berarti jumlah cluster tersebut langsung dipilih sebagai model akhir. Hal ini karena pembagian properti menjadi dua kelompok saja berpotensi menghasilkan pengelompokan yang terlalu umum dan kurang menggambarkan keragaman harga serta karakteristik rumah di Jabodetabek.

Sementara itu, K=4 yang diperoleh dari metode Elbow berpotensi memberikan pengelompokan yang lebih detail. Dengan pembagian tersebut, calon pembeli dapat lebih mudah membandingkan properti berdasarkan harga, luas tanah, luas bangunan, jumlah kamar tidur, jumlah kamar mandi, dan jumlah lantai.

Oleh karena itu, pemilihan model akhir tidak hanya dilihat dari nilai Silhouette Score tertinggi, tetapi juga perlu mempertimbangkan karakteristik setiap cluster dan manfaat hasil pengelompokan bagi pengguna aplikasi.

### **3. Apa arti nilai Silhouette Score yang diperoleh dalam konteks kasus kelompok?**

Nilai Silhouette Score tertinggi yang diperoleh adalah 0,4834 pada K=2. Berdasarkan pedoman interpretasi dalam Manual Pertemuan 13, nilai tersebut berada pada rentang 0,25–0,50, yang berarti struktur cluster masih tergolong lemah, tetapi dapat dipertanggungjawabkan.

Dalam kasus kelompok kami, hasil tersebut menunjukkan bahwa properti rumah di Jabodetabek dapat dikelompokkan berdasarkan kesamaan harga dan karakteristik fisiknya. Namun, pemisahan antarcluster masih belum terlalu jelas karena terdapat beberapa properti pada cluster yang berbeda, tetapi memiliki karakteristik yang hampir sama.

Artinya, algoritma K-Means sudah mampu menemukan pola pengelompokan pada dataset properti yang digunakan, meskipun hasil pemisahan antarcluster belum maksimal.

### **Kesimpulan Varian 2**

Berdasarkan eksperimen algoritma K-Means dengan sembilan variasi nilai K, mulai dari K=2 hingga K=10, diperoleh hasil bahwa K=2 memiliki nilai Silhouette Score tertinggi sebesar 0,4834 dan Davies-Bouldin Index terendah sebesar 0,8806.

Hasil tersebut berbeda dengan Varian 1 yang menggunakan metode Elbow dan menghasilkan K=4. Dari perbedaan ini, dapat diketahui bahwa menentukan jumlah cluster terbaik tidak cukup hanya berdasarkan satu metode penilaian, tetapi juga perlu mempertimbangkan karakteristik data dan tujuan pengelompokan.

Pada Varian 2, K=2 dipilih sebagai jumlah cluster terbaik berdasarkan Silhouette Score dan akan digunakan dalam eksperimen pemilihan fitur pada Varian 3. Sementara itu, penentuan model akhir akan dilakukan dengan membandingkan hasil dari ketiga varian eksperimen.

## 3.4 Unduh, commit, dan teruskan

In [ ]:
# Unduh hasil Bagian 3, lalu commit ke repositori dengan akun GitHub
# ANDA SENDIRI. Riwayat commit adalah bukti penilaian perorangan.
from google.colab import files
files.download('data/hasil_varian/varian2.json')
files.download('gambar/varian2_silhouette.png')
print()
print('Langkah berikutnya:')
print('  1. Commit kedua berkas di atas dengan akun GitHub Anda sendiri.')
print('  2. File -> Download -> Download .ipynb, beri nama 03_varian_2.ipynb,')
print('     lalu commit ke folder notebook/.')
print('  3. Teruskan `varian2.json` ke Anggota 3.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Langkah berikutnya:
  1. Commit kedua berkas di atas dengan akun GitHub Anda sendiri.
  2. File -> Download -> Download .ipynb, beri nama 03_varian_2.ipynb,
     lalu commit ke folder notebook/.
  3. Teruskan `varian2.json` ke Anggota 3.


## Checklist Bagian 3

- [✓] `varian2.json` ter-commit di `data/hasil_varian/`
- [✓] `varian2_silhouette.png` ter-commit di `gambar/`
- [✓] Notebook ini diunduh, diberi nama `03_varian_2.ipynb`, dan ter-commit di `notebook/`
- [✓] Commit dilakukan dengan akun GitHub Anggota 2 sendiri
- [✓] Perbandingan dengan kesimpulan Elbow sudah ditulis
- [✓] `varian2.json` sudah dikirim ke Anggota 3

# Bagian 4 — Varian 3 — Pemilihan Fitur dan Profil Klaster

**Proyek Akhir Pembelajaran Mesin (INF62325) · Cabang C — Klastering K-Means**
**S1 Informatika · UIN Siber Syekh Nurjati Cirebon**

| | |
|---|---|
| Pertemuan | dikerjakan setelah Pertemuan 12, dipresentasikan Pertemuan 14 |
| Tahap resmi | Tahap 2 — Varian Eksperimen 3 |
| Dikerjakan oleh | **Anggota 3** saja |
| Berkas yang dibutuhkan | `data_bersih.csv` dari Bagian 1 dan `varian2.json` dari Anggota 2 |
| Luaran wajib | `data/hasil_varian/varian3.json`, `gambar/varian3_pemilihan_fitur.png`, tabel profil segmen |
| Dinilai pada | Progres 2, Pertemuan 14, **35% perorangan** |
| Tenggat | selesai sebelum Pertemuan 14 |

| | |
|---|---|
| Kelompok | _(isi)_ |
| Paket dataset | _(isi sesuai penetapan dosen)_ |
| Nama dan NIM | _(isi)_ |

---

## Panduan Bagian 4

> **Ketentuan Panduan Proyek Akhir, Cabang C, Varian 3:** *Bandingkan hasil klaster dari dua himpunan fitur berbeda, lalu susun profil tiap klaster.*

Bagian ini dikerjakan **Anggota 3 saja**, dan dinilai **perorangan**. Bagian ini juga yang menentukan himpunan fitur untuk model final di Bagian 6, jadi hasilnya dipakai langsung oleh aplikasi web.

**Urutan kerja**

1. Jalankan 4.0 (persiapan).
2. Jalankan 4.1. Anda membutuhkan `data_bersih.csv` dan `varian2.json`.
3. Jalankan 4.2.
4. **Tulis sendiri** jawaban di 4.3, termasuk mengisi tabel nama segmen. Tabel itu dipakai kembali di Bagian 6.
5. Jalankan 4.4 dan commit dengan akun Anda sendiri.

**Yang perlu dipahami lebih dulu**

Bagian ini **tidak mengubah k**. Nilai k diambil dari hasil Bagian 3. Kedua himpunan fitur diuji pada k yang sama, supaya yang dibandingkan benar-benar pengaruh pemilihan fitur, bukan pengaruh jumlah klaster.

Himpunan A adalah seluruh fitur. Himpunan B disusun dengan dua saringan:

1. Dari setiap pasangan fitur dengan korelasi di atas 0,9, salah satunya dibuang — karena bagi K-Means dua fitur kembar berarti satu dimensi mendapat bobot ganda.
2. Fitur yang nilainya hampir sama di semua klaster dibuang — karena fitur seperti itu tidak membantu membedakan segmen, tetapi tetap menambah jarak.

Setelah salah satu himpunan terpilih, bagian kedua ketentuan dikerjakan: **menyusun profil tiap klaster**. Profil dibaca dari nilai rata-rata tiap fitur per klaster, dinyatakan relatif terhadap rata-rata keseluruhan.

**Aturan penting dalam memberi nama segmen.** Nama harus dapat ditelusuri ke kolom datanya. "Pengguna saldo besar dengan pembayaran minimum" sah, karena saldo dan pembayaran minimum adalah kolom yang ada. "Pelanggan loyal" tidak sah, karena tidak ada kolom kesetiaan di dalam dataset — nama itu menyimpulkan hal yang tidak diukur.

---

## 4.1 Mengambil berkas yang diperlukan

Sel ini memeriksa berkas yang Anda butuhkan. Yang sudah ada dilewati; yang belum ada akan diminta untuk diunggah. Aman dijalankan berulang kali.

In [ ]:
perlu('data/data_bersih.csv', 'data/hasil_varian/varian2.json')

Lengkap: data/data_bersih.csv, data/hasil_varian/varian2.json


## 4.2 Menjalankan varian 3

In [ ]:
from __future__ import annotations

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import davies_bouldin_score, silhouette_score

from alat import (SEED, baca_hasil, muat_bersih, nama_otomatis, profil_relatif,
                  simpan_hasil, skalakan)

class args:
    k = 0                   # 0 berarti ikut hasil Bagian 3
    ambang_korelasi = 0.9   # pasangan fitur di atas ini dianggap kembar
    ambang_beda = 0.25      # fitur di bawah ini dianggap tidak membedakan

print('=' * 70)
print('VARIAN 3 - PEMILIHAN FITUR DAN PROFIL KLASTER')
print('=' * 70)

df = muat_bersih()
semua = df.columns.tolist()

v2 = baca_hasil(2)
if args.k:
    K, asal_k = args.k, 'ditetapkan lewat argumen --k'
elif v2:
    K, asal_k = v2['k_terpilih'], 'mengikuti hasil Varian 2'
else:
    K, asal_k = 4, 'nilai bawaan, Varian 2 belum dijalankan'
print(f'Jumlah klaster dipakai: k = {K}  ({asal_k})')
print('Kedua himpunan fitur diuji pada k yang SAMA, agar yang dibandingkan')
print('benar-benar pengaruh pemilihan fitur, bukan pengaruh perbedaan k.\n')


def uji(fitur: list[str]) -> dict:
    sub = df[fitur]
    X, _ = skalakan(sub)
    km = KMeans(n_clusters=K, init='k-means++', n_init=20, random_state=SEED)
    lab = km.fit_predict(X)
    return {
        'fitur': fitur,
        'jumlah_fitur': len(fitur),
        'silhouette': float(silhouette_score(X, lab)),
        'davies_bouldin': float(davies_bouldin_score(X, lab)),
        'inertia': float(km.inertia_),
        'label': lab,
        'X': X,
        'sub': sub,
    }


# ---------- Himpunan A: seluruh fitur ----------
A = uji(semua)

# ---------- Menyusun Himpunan B dengan dua saringan ----------
print('-' * 70)
print('Saringan 1 - membuang salah satu dari pasangan fitur yang kembar')
kor = df.corr().abs()
buang_kembar = set()
for i, a in enumerate(semua):
    for b in semua[i + 1:]:
        if a in buang_kembar or b in buang_kembar:
            continue
        if kor.loc[a, b] > args.ambang_korelasi:
            # buang yang rata-rata korelasinya ke fitur lain lebih tinggi
            korban = a if kor[a].mean() >= kor[b].mean() else b
            buang_kembar.add(korban)
            print(f'  {a} vs {b}: r={kor.loc[a, b]:.3f} -> buang {korban}')
if not buang_kembar:
    print(f'  tidak ada pasangan dengan korelasi di atas {args.ambang_korelasi}')

print('')
print('Saringan 2 - membuang fitur yang nilainya hampir sama di semua klaster')
profil_A = profil_relatif(A['sub'], A['label'])
beda_maks = profil_A.abs().max(axis=1).sort_values()
buang_datar = [f for f, v in beda_maks.items() if v < args.ambang_beda]
for f in buang_datar:
    print(f'  {f:<34} simpangan maksimum {beda_maks[f]:.2f} sd -> dibuang')
if not buang_datar:
    print(f'  tidak ada fitur dengan simpangan di bawah {args.ambang_beda} sd')

fitur_B = [f for f in semua if f not in buang_kembar and f not in buang_datar]
if len(fitur_B) < 3:
    print('\n  Saringan menyisakan terlalu sedikit fitur. '
          'Himpunan B dikembalikan ke seluruh fitur kecuali yang kembar.')
    fitur_B = [f for f in semua if f not in buang_kembar]

B = uji(fitur_B)

# ---------- Perbandingan ----------
print('\n' + '=' * 70)
print('PERBANDINGAN DUA HIMPUNAN FITUR')
print('=' * 70)
print(f'{"":<24}{"Himpunan A":>14}{"Himpunan B":>14}')
print(f'{"Jumlah fitur":<24}{A["jumlah_fitur"]:>14}{B["jumlah_fitur"]:>14}')
print(f'{"Silhouette (utama)":<24}{A["silhouette"]:>14.4f}{B["silhouette"]:>14.4f}')
print(f'{"Davies-Bouldin":<24}{A["davies_bouldin"]:>14.4f}{B["davies_bouldin"]:>14.4f}')
print(f'{"Inertia":<24}{A["inertia"]:>14.2f}{B["inertia"]:>14.2f}')

selisih = B['silhouette'] - A['silhouette']
terbaik, nama_terbaik = (B, 'B') if selisih > 0 else (A, 'A')
print(f'\nSelisih Silhouette B dikurangi A: {selisih:+.4f}')
print(f'Himpunan terpilih: {nama_terbaik} ({terbaik["jumlah_fitur"]} fitur)')

if abs(selisih) < 0.01:
    catatan = ('Selisihnya di bawah 0,01 sehingga kedua himpunan praktis '
               'setara. Himpunan dengan fitur lebih sedikit tetap lebih '
               'disukai karena aplikasi webnya meminta lebih sedikit isian '
               'kepada pengguna.')
    terbaik, nama_terbaik = B, 'B'
elif selisih > 0:
    catatan = ('Membuang fitur yang tidak membedakan klaster menaikkan mutu '
               'pemisahan. Fitur semacam itu hanya menambah dimensi tanpa '
               'menambah informasi, sehingga melemahkan perhitungan jarak.')
else:
    catatan = ('Himpunan lengkap justru lebih baik. Fitur yang disaring '
               'ternyata masih menyumbang pembeda, sehingga penyaringan '
               'tidak diteruskan.')
print(f'Catatan: {catatan}')

# ---------- Profil tiap klaster pada himpunan terpilih ----------
print('\n' + '=' * 70)
print(f'PROFIL TIAP KLASTER - HIMPUNAN {nama_terbaik}')
print('=' * 70)
profil = profil_relatif(terbaik['sub'], terbaik['label'])
print('Satuan: simpangan baku terhadap rata-rata seluruh data.')
print('Nilai di atas +0,5 jauh lebih tinggi, di bawah -0,5 jauh lebih rendah.\n')
print(profil.to_string())

jumlah = {str(int(c)): int((terbaik['label'] == c).sum())
          for c in np.unique(terbaik['label'])}
print('\nJumlah anggota:')
for c, n in jumlah.items():
    print(f'  Klaster {c}: {n:>5} baris ({n / len(df) * 100:5.1f}%)')

nama = nama_otomatis(profil)
print('\nUsulan nama otomatis - WAJIB diganti dengan nama yang bermakna:')
for c, nm in nama.items():
    print(f'  Klaster {c}: {nm}')

# ---------- Gambar ----------
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].bar(['A\nlengkap', 'B\ndisaring'], [A['silhouette'], B['silhouette']],
          color=['tab:blue', 'tab:green'])
ax[0].set_ylabel('Silhouette Score')
ax[0].set_title(f'Varian 3 - pengaruh pemilihan fitur (k={K})')
for i, v in enumerate([A['silhouette'], B['silhouette']]):
    ax[0].text(i, v, f'{v:.4f}', ha='center', va='bottom')

koor = PCA(n_components=2, random_state=SEED).fit_transform(terbaik['X'])
for c in np.unique(terbaik['label']):
    m = terbaik['label'] == c
    ax[1].scatter(koor[m, 0], koor[m, 1], s=12, alpha=0.6, label=f'Klaster {c}')
ax[1].set_title(f'Sebaran klaster, himpunan {nama_terbaik}')
ax[1].set_xlabel('PC1'); ax[1].set_ylabel('PC2'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig('gambar/varian3_pemilihan_fitur.png', dpi=110)
plt.close()

jalur = simpan_hasil(3, {
    'varian': 3,
    'nama': 'Pemilihan fitur dan profil klaster',
    'yang_diubah': f'Himpunan fitur: A {A["jumlah_fitur"]} fitur '
                   f'dibandingkan B {B["jumlah_fitur"]} fitur, pada k={K}',
    'k_dipakai': K,
    'asal_k': asal_k,
    'himpunan_A': {'fitur': A['fitur'], 'silhouette': round(A['silhouette'], 4),
                   'davies_bouldin': round(A['davies_bouldin'], 4),
                   'inertia': round(A['inertia'], 2)},
    'himpunan_B': {'fitur': B['fitur'], 'silhouette': round(B['silhouette'], 4),
                   'davies_bouldin': round(B['davies_bouldin'], 4),
                   'inertia': round(B['inertia'], 2)},
    'dibuang_kembar': sorted(buang_kembar),
    'dibuang_datar': buang_datar,
    'himpunan_terpilih': nama_terbaik,
    'fitur_terpilih': terbaik['fitur'],
    'k_terpilih': K,
    'metrik_utama': {'silhouette': round(terbaik['silhouette'], 4)},
    'metrik_pendukung': {'inertia': round(terbaik['inertia'], 2),
                         'davies_bouldin': round(terbaik['davies_bouldin'], 4)},
    'catatan': catatan,
    'jumlah_anggota': jumlah,
    'profil_relatif': {str(c): {f: float(v) for f, v in profil[c].items()}
                       for c in profil.columns},
    'nama_otomatis': nama,
})
print(f'\nTersimpan: {jalur}  dan  gambar/varian3_pemilihan_fitur.png')
print('Commit berkas ini dengan akun GitHub Anda sendiri.')

VARIAN 3 - PEMILIHAN FITUR DAN PROFIL KLASTER
Jumlah klaster dipakai: k = 4  (mengikuti hasil Varian 2)
Kedua himpunan fitur diuji pada k yang SAMA, agar yang dibandingkan
benar-benar pengaruh pemilihan fitur, bukan pengaruh perbedaan k.

----------------------------------------------------------------------
Saringan 1 - membuang salah satu dari pasangan fitur yang kembar
  tidak ada pasangan dengan korelasi di atas 0.9

Saringan 2 - membuang fitur yang nilainya hampir sama di semua klaster
  TENURE                             simpangan maksimum 0.05 sd -> dibuang

PERBANDINGAN DUA HIMPUNAN FITUR
                            Himpunan A    Himpunan B
Jumlah fitur                        17            16
Silhouette (utama)              0.4096        0.4504
Davies-Bouldin                  1.1515        1.0482
Inertia                       18097.21      15099.99

Selisih Silhouette B dikurangi A: +0.0408
Himpunan terpilih: B (16 fitur)
Catatan: Membuang fitur yang tidak membedakan klaster me

## 4.3 Yang harus Anda tulis sendiri

Rubrik varian perorangan menilai tiga hal: **ketepatan implementasi**, **kebenaran evaluasi**, dan **kedalaman interpretasi**. Butir ketiga tidak dapat dihasilkan oleh kode, dan di situlah nilai paling banyak hilang.

1. Fitur apa saja yang disaring, dan mengapa pantas disaring? Sebut nama kolomnya dan alasannya satu per satu.
2. Apakah penyaringan menaikkan atau menurunkan Silhouette? Apa artinya bagi kelompok Anda?
3. **Beri nama tiap segmen** berdasarkan angka profil di atas, dengan aturan nama yang dapat ditelusuri ke kolomnya.

Isi tabel berikut. Tabel ini dipakai lagi di Bagian 6 dan muncul di aplikasi web, jadi isi dengan serius.

| Klaster | Jumlah anggota | Fitur menonjol tinggi | Fitur menonjol rendah | Nama segmen | Tindakan yang disarankan |
|---|---|---|---|---|---|
| 0 | | | | | |
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

_(jawab di sini — klik dua kali pada sel untuk menyuntingnya)_

## 4.4 Unduh, commit, dan teruskan

In [ ]:
# Unduh hasil Bagian 4, lalu commit ke repositori dengan akun GitHub
# ANDA SENDIRI. Riwayat commit adalah bukti penilaian perorangan.
from google.colab import files
files.download('data/hasil_varian/varian3.json')
files.download('gambar/varian3_pemilihan_fitur.png')
print()
print('Langkah berikutnya:')
print('  1. Commit kedua berkas di atas dengan akun GitHub Anda sendiri.')
print('  2. File -> Download -> Download .ipynb, beri nama 03_varian_3.ipynb,')
print('     lalu commit ke folder notebook/.')
print('  3. Teruskan `varian3.json` ke kelompok untuk Bagian 5.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Langkah berikutnya:
  1. Commit kedua berkas di atas dengan akun GitHub Anda sendiri.
  2. File -> Download -> Download .ipynb, beri nama 03_varian_3.ipynb,
     lalu commit ke folder notebook/.
  3. Teruskan `varian3.json` ke kelompok untuk Bagian 5.


## Checklist Bagian 4

- [ ] `varian3.json` ter-commit di `data/hasil_varian/`
- [ ] `varian3_pemilihan_fitur.png` ter-commit di `gambar/`
- [ ] Notebook ini diunduh, diberi nama `03_varian_3.ipynb`, dan ter-commit di `notebook/`
- [ ] Commit dilakukan dengan akun GitHub Anggota 3 sendiri
- [ ] Tabel nama segmen sudah terisi lengkap, bukan nama otomatis
- [ ] Setiap nama segmen dapat ditelusuri ke kolom datanya
- [ ] `varian3.json` sudah diteruskan ke kelompok

# Bagian 5 — Tabel Perbandingan Tiga Varian

**Proyek Akhir Pembelajaran Mesin (INF62325) · Cabang C — Klastering K-Means**
**S1 Informatika · UIN Siber Syekh Nurjati Cirebon**

| | |
|---|---|
| Pertemuan | Pertemuan 14 (Progres 2) |
| Tahap resmi | Tahap 2 — Perbandingan dan Pemilihan Model |
| Dikerjakan oleh | kelompok |
| Berkas yang dibutuhkan | ketiga berkas `varian1.json`, `varian2.json`, `varian3.json` |
| Luaran wajib | `data/tabel_perbandingan.md` dan satu paragraf kesimpulan |
| Dinilai pada | Progres 2, Pertemuan 14, **10%** (nilai kelompok) |
| Tenggat | selesai sebelum Pertemuan 14 |

| | |
|---|---|
| Kelompok | _(isi)_ |
| Paket dataset | _(isi sesuai penetapan dosen)_ |
| Nama dan NIM | _(isi)_ |

---

## Panduan Bagian 5

Tabel ini adalah bukti utama bahwa kelompok benar-benar bereksperimen, bukan menjalankan satu kode lalu menyebutnya tiga varian. Karena itu tabelnya harus dibangun dari **ketiga berkas json** yang dihasilkan masing-masing anggota, bukan ditulis ulang dengan tangan.

**Urutan kerja**

1. Pastikan ketiga anggota sudah menyelesaikan Bagian 2, 3, dan 4 serta sudah commit berkas json-nya.
2. Jalankan 5.0 (persiapan).
3. Jalankan 5.1. Bila ada json yang belum ada, Colab akan memintanya.
4. Jalankan 5.2 untuk menyusun tabel.
5. **Tulis sendiri** paragraf kesimpulan di 5.3. Paragraf ini menentukan varian mana yang dipakai aplikasi web.
6. Tempelkan tabel dan paragraf kesimpulan ke `README.md`, lalu commit.

**Yang perlu dipahami lebih dulu**

Ketiga varian **tidak dinilai dari siapa yang angkanya paling bagus**. Varian 1 dan 2 menjawab pertanyaan berapa k, Varian 3 menjawab pertanyaan fitur mana. Jadi tabel ini bukan lomba, melainkan dasar untuk satu keputusan: himpunan fitur dan nilai k mana yang dipakai model final.

Tiga metrik yang dilaporkan:

| Metrik | Arah yang lebih baik | Yang diukur |
|---|---|---|
| Silhouette Score | makin besar | seberapa terpisah antarklaster |
| Davies-Bouldin Index | makin kecil | rasio sebaran dalam klaster terhadap jarak antarklaster |
| Inertia | makin kecil, tetapi hanya pada k yang sama | jumlah kuadrat jarak ke pusat klaster |

Inertia **tidak boleh** dibandingkan antar-k yang berbeda, karena nilainya selalu turun saat k naik. Kesalahan ini sering muncul pada tabel perbandingan mahasiswa.

---

## 5.1 Mengambil berkas yang diperlukan

Kumpulkan ketiga berkas json dari anggota lain lebih dulu. Kalau salah satu belum ada, Bagian ini tidak dapat diselesaikan — dan itu berarti ada anggota yang belum menyelesaikan variannya.

In [ ]:
perlu('data/data_bersih.csv', 'data/hasil_varian/varian1.json',
      'data/hasil_varian/varian2.json', 'data/hasil_varian/varian3.json')

Lengkap: data/data_bersih.csv, data/hasil_varian/varian1.json, data/hasil_varian/varian2.json, data/hasil_varian/varian3.json


## 5.2 Menyusun tabel

In [ ]:
from __future__ import annotations

import pandas as pd

from alat import baca_hasil

print('=' * 70)
print('TABEL PERBANDINGAN TIGA VARIAN')
print('=' * 70)

hasil = {n: baca_hasil(n) for n in (1, 2, 3)}
hilang = [n for n, h in hasil.items() if h is None]
if hilang:
    print('Varian berikut belum dijalankan: ' +
          ', '.join(str(n) for n in hilang))
    print('Jalankan skripnya lebih dulu, lalu ulangi perintah ini.')
    raise SystemExit(1)

baris = []
for n in (1, 2, 3):
    h = hasil[n]
    pendukung = ', '.join(f'{k.replace("_", " ")} {v}'
                          for k, v in h['metrik_pendukung'].items())
    if n == 1:
        catatan = f'Titik siku pada k={h["k_terpilih"]}'
    elif n == 2:
        catatan = ('Sama dengan kesimpulan Elbow'
                   if h.get('berbeda_dari_elbow') is False
                   else f'Berbeda dari Elbow (k={h.get("k_menurut_elbow")})')
    else:
        catatan = (f'Himpunan {h["himpunan_terpilih"]} terpilih, '
                   f'{len(h["fitur_terpilih"])} fitur')
    baris.append({
        'Varian': n,
        'Apa yang diubah': h['yang_diubah'],
        'k': h['k_terpilih'],
        'Silhouette (utama)': h['metrik_utama']['silhouette'],
        'Metrik pendukung': pendukung,
        'Catatan': catatan,
    })

tabel = pd.DataFrame(baris)
print(tabel.to_string(index=False))

# --- Varian mana yang dipakai untuk aplikasi web ---
terbaik = max((1, 2, 3), key=lambda n: hasil[n]['metrik_utama']['silhouette'])
print(f'\nSilhouette tertinggi ada pada Varian {terbaik} '
      f'({hasil[terbaik]["metrik_utama"]["silhouette"]})')
print('Varian 3 selalu menjadi acuan susunan fitur aplikasi, karena hanya')
print('varian itu yang menguji himpunan fitur. Varian 1 dan 2 menguji k.')

k_final = hasil[3]['k_terpilih']
fitur_final = hasil[3]['fitur_terpilih']
print(f'\nUsulan konfigurasi model final: k={k_final}, '
      f'{len(fitur_final)} fitur dari Varian 3.')

# --- Berkas markdown untuk README ---
md = ['## Tabel perbandingan tiga varian', '',
      '| Varian | Apa yang diubah | k | Silhouette (utama) | Metrik pendukung | Catatan |',
      '| --- | --- | --- | --- | --- | --- |']
for r in baris:
    md.append(f'| {r["Varian"]} | {r["Apa yang diubah"]} | {r["k"]} | '
              f'{r["Silhouette (utama)"]} | {r["Metrik pendukung"]} | '
              f'{r["Catatan"]} |')
md += ['', '### Kesimpulan', '',
       '_Paragraf ini WAJIB ditulis sendiri oleh kelompok dan menjawab tiga hal:_',
       '',
       f'1. Varian mana yang dipakai untuk aplikasi web. _(usulan: Varian 3, '
       f'k={k_final}, {len(fitur_final)} fitur)_',
       '2. Mengapa varian itu dipilih.',
       '3. Apa yang dikorbankan dengan memilihnya.',
       '',
       '_Jawaban berupa varian dengan angka tertinggi, tanpa alasan lain, '
       'belum memadai._']

with open('data/tabel_perbandingan.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(md))
tabel.to_csv('data/tabel_perbandingan.csv', index=False)

print('\nTersimpan: data/tabel_perbandingan.md (tempel ke README.md)')
print('           data/tabel_perbandingan.csv')

TABEL PERBANDINGAN TIGA VARIAN
 Varian                                              Apa yang diubah  k  Silhouette (utama)                        Metrik pendukung                       Catatan
      1      Jumlah klaster ditentukan dari titik siku kurva inertia  4              0.4096                        inertia 18097.21           Titik siku pada k=4
      2          Jumlah klaster ditentukan dari Silhouette tertinggi  4              0.4096 inertia 18097.21, davies bouldin 1.1515  Sama dengan kesimpulan Elbow
      3 Himpunan fitur: A 17 fitur dibandingkan B 16 fitur, pada k=4  4              0.4504 inertia 15099.99, davies bouldin 1.0482 Himpunan B terpilih, 16 fitur

Silhouette tertinggi ada pada Varian 3 (0.4504)
Varian 3 selalu menjadi acuan susunan fitur aplikasi, karena hanya
varian itu yang menguji himpunan fitur. Varian 1 dan 2 menguji k.

Usulan konfigurasi model final: k=4, 16 fitur dari Varian 3.

Tersimpan: data/tabel_perbandingan.md (tempel ke README.md)
           data/

## 5.3 Paragraf kesimpulan, wajib ditulis sendiri

Paragraf ini menjawab tiga hal:

1. **Varian mana yang dipakai untuk aplikasi web?** Sebut nilai k dan himpunan fiturnya.
2. **Mengapa varian itu dipilih?** Sebut metrik dan pertimbangan praktisnya.
3. **Apa yang dikorbankan dengan memilihnya?** Setiap pilihan punya konsekuensi.

Jawaban berupa "varian 3 dipilih karena Silhouette-nya paling tinggi", tanpa alasan lain, **belum memadai**. Nilai tertinggi saja bukan alasan: harus dijelaskan mengapa angka itu berarti dalam konteks kasus kelompok Anda, dan apa yang hilang dari pilihan itu.

_(tulis di sini — klik dua kali pada sel untuk menyuntingnya)_

## 5.4 Unduh dan commit

In [ ]:
from google.colab import files
files.download('data/tabel_perbandingan.md')
print()
print('Langkah berikutnya:')
print('  1. Tempelkan isi tabel_perbandingan.md ke README.md.')
print('  2. Tambahkan paragraf kesimpulan 5.3 di bawah tabel itu.')
print('  3. Commit keduanya ke repositori kelompok.')
print('  4. Unduh notebook ini sebagai 04_perbandingan.ipynb, commit ke notebook/.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Langkah berikutnya:
  1. Tempelkan isi tabel_perbandingan.md ke README.md.
  2. Tambahkan paragraf kesimpulan 5.3 di bawah tabel itu.
  3. Commit keduanya ke repositori kelompok.
  4. Unduh notebook ini sebagai 04_perbandingan.ipynb, commit ke notebook/.


## Checklist Bagian 5

- [ ] Tabel dibangun dari ketiga berkas json, bukan ditulis ulang dengan tangan
- [ ] `data/tabel_perbandingan.md` ter-commit
- [ ] Tabel sudah ditempel di `README.md`
- [ ] Paragraf kesimpulan sudah ditulis dan menjawab ketiga hal di atas
- [ ] Inertia tidak dibandingkan antar-k yang berbeda
- [ ] Nilai k dan himpunan fitur untuk model final sudah ditetapkan